# Notebook 05: Structured Data Extraction from Scientific Literature

**CABS AI Productivity Series**
Workshop: *LangChain, LangGraph & Local LLM Deployment: Building AI Agent Systems That Keep Your Data Safe*

---

## The Problem

Imagine you are starting a project on kinase inhibitors. Your manager asks you to compile a table of all reported IC50 values for your target across published papers. Manually, this means:

1. Read 50+ abstracts
2. Copy drug name, target, IC50, cell line, and assay type into a spreadsheet
3. Repeat for every paper

**An LLM can do this — but by default it returns prose, not a table.**

This notebook teaches you to extract **structured, machine-readable data** from text using Pydantic schemas. You will process a batch of 8 kinase inhibitor abstracts and export the results to a pandas DataFrame in under 2 minutes.

---

## What You Will Learn

1. **Pydantic schemas** — describe the exact structure you want the LLM to return
2. **`with_structured_output()`** — force the LLM to return JSON matching your schema
3. **Few-shot prompting** — give examples to improve extraction accuracy
4. **Batch processing** — run extraction over many abstracts programmatically
5. **DataFrame export** — pipe results straight into pandas for filtering and analysis

---

## Data Privacy Note

Every abstract you send goes to Google Gemini's servers. For published literature this is acceptable.
For unpublished assay data from your own lab, use the local Ollama pattern from Notebook 03.


# Setup: Get Your Free Gemini API Key

1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with your Google account → Accept Terms of Service
3. Left sidebar → **Get API Key** → **Create API key**
4. Copy the key (starts with `AIza...`)

No credit card needed.


In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# langchain              - framework for building LLM applications
# langchain-google-genai - connects LangChain to Google Gemini
# pydantic               - data validation and schema definition
# pandas                 - tabular data manipulation

!pip install -q langchain langchain-google-genai pydantic pandas
print("✅ Packages installed!")


In [ ]:
# ============================================================
# STEP 2: Enter your Gemini API key
# ============================================================

import getpass
import os

api_key = getpass.getpass("Paste your Gemini API key here: ")
os.environ["GOOGLE_API_KEY"] = api_key
print("✅ API key set!")


In [ ]:
# ============================================================
# STEP 3: Quick test — make sure Gemini is working
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke("What does IC50 measure? One sentence.")
print(response.content)
print("\n✅ Gemini is working!")


---
## The Problem with Unstructured Output

Ask an LLM to extract IC50 data without a schema and you get readable prose —
but prose is hard to filter, sort, or combine with other data.


In [ ]:
# ============================================================
# STEP 4: Show the problem — unstructured extraction
# ============================================================
# A useful answer, but impossible to load directly into a DataFrame.

abstract = (
    "Erlotinib is a small-molecule EGFR inhibitor. In A549 cells, a 72-hour "
    "CellTiter-Glo assay showed an IC50 of 2.1 nM. Western blot confirmed "
    "EGFR phosphorylation suppression above 10 nM."
)

response = llm.invoke(
    f"Extract the drug name, target, IC50, cell line, and assay type from this abstract:\n\n{abstract}"
)
print("Unstructured output:")
print(response.content)
print()
print("Type:", type(response.content))
print("\n⚠️  This is a string — you cannot filter or sort it as a table.")


---
## The Solution: Pydantic Structured Output

With `with_structured_output()`, you define a **schema** (a Pydantic model) and the LLM
is forced to return data that matches it exactly — as a Python object, not prose.

Think of the schema as a form the LLM must fill in.


In [ ]:
# ============================================================
# STEP 5: Define the extraction schema with Pydantic
# ============================================================
# Each field has a type annotation and a description.
# The description is what guides the LLM to fill that field correctly.

from pydantic import BaseModel, Field
from typing import Optional


class DrugActivityRecord(BaseModel):
    """Structured record of drug activity extracted from a scientific abstract."""

    drug_name: str = Field(description="Name of the drug or compound (e.g. erlotinib, imatinib)")
    target: str = Field(description="Primary molecular target (gene/protein) being inhibited or modulated")
    ic50_value: Optional[float] = Field(
        default=None,
        description="Numeric IC50 value. Extract the number only, no units."
    )
    ic50_unit: Optional[str] = Field(
        default=None,
        description="Unit of IC50 measurement: nM, μM, uM, pM, etc."
    )
    cell_line: Optional[str] = Field(
        default=None,
        description="Cell line used in the assay (e.g. A549, K562, MCF-7). Null if biochemical assay."
    )
    assay_type: Optional[str] = Field(
        default=None,
        description="Type of assay: CellTiter-Glo, BrdU, colony formation, biochemical kinase assay, etc."
    )
    key_finding: str = Field(
        description="One sentence summarising the most important biological finding in the abstract."
    )


print("✅ Schema defined: DrugActivityRecord")
print("   Fields:", list(DrugActivityRecord.model_fields.keys()))


In [ ]:
# ============================================================
# STEP 6: Create the structured extraction chain
# ============================================================
# .with_structured_output() wraps the LLM with a parser that
# validates the response against your Pydantic schema.

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a scientific data extraction assistant. "
     "Extract the requested fields from the provided abstract. "
     "If a field is not mentioned in the abstract, return null for optional fields. "
     "Be precise — copy numeric values exactly as written."),
    ("human", "Abstract:\n\n{abstract}"),
])

structured_llm = llm.with_structured_output(DrugActivityRecord)
extraction_chain = prompt | structured_llm

print("✅ Extraction chain ready!")
print("   Input  → abstract text")
print("   Output → DrugActivityRecord Python object")


In [ ]:
# ============================================================
# STEP 7: Test extraction on a single abstract
# ============================================================

test_abstract = (
    "Erlotinib is a small-molecule EGFR inhibitor. In A549 non-small cell lung cancer cells, "
    "a 72-hour CellTiter-Glo cell viability assay showed an IC50 of 2.1 nM. Western blot "
    "confirmed complete EGFR phosphorylation suppression above 10 nM."
)

record = extraction_chain.invoke({"abstract": test_abstract})

print("Extracted record:")
print(f"  Drug name:   {record.drug_name}")
print(f"  Target:      {record.target}")
print(f"  IC50:        {record.ic50_value} {record.ic50_unit}")
print(f"  Cell line:   {record.cell_line}")
print(f"  Assay type:  {record.assay_type}")
print(f"  Key finding: {record.key_finding}")
print()
print("Python type:", type(record))
print("\n✅ Structured object — ready for a DataFrame!")


---
## Batch Processing

Now we scale up: run the same extraction over 8 kinase inhibitor abstracts.
Each returns a `DrugActivityRecord` object. We collect them all into a list and
convert to a pandas DataFrame.


In [ ]:
# ============================================================
# STEP 8: Define the batch of 8 abstracts
# ============================================================

abstracts = [
    {
        "id": 1,
        "text": (
            "Erlotinib is a small-molecule inhibitor of the epidermal growth factor receptor (EGFR) tyrosine kinase. "
            "In this study, we evaluated its antiproliferative activity in the A549 non-small cell lung cancer cell line "
            "using a 72-hour CellTiter-Glo cell viability assay. Erlotinib demonstrated potent growth inhibition with "
            "an IC50 of 2.1 nM. Western blot analysis confirmed complete suppression of EGFR phosphorylation at "
            "concentrations above 10 nM."
        ),
    },
    {
        "id": 2,
        "text": (
            "We investigated the activity of imatinib mesylate against BCR-ABL kinase in chronic myelogenous leukemia. "
            "Using a K562 CML cell line in a 96-hour BrdU proliferation assay, imatinib showed an IC50 of 0.18 μM. "
            "Flow cytometry demonstrated G1 phase arrest consistent with BCR-ABL target engagement. "
            "Selectivity profiling against a 50-kinase panel revealed >100-fold selectivity for ABL over most off-targets."
        ),
    },
    {
        "id": 3,
        "text": (
            "Vemurafenib (PLX4032) was profiled against BRAF V600E-mutant melanoma. In the A375 cell line, "
            "a 5-day colony formation assay yielded an IC50 of 31 nM. Biochemical kinase assays confirmed direct "
            "BRAF V600E inhibition with a Ki of 13 nM. The compound showed >100-fold selectivity over wild-type BRAF "
            "in biochemical assays, supporting its mutation-selective therapeutic window."
        ),
    },
    {
        "id": 4,
        "text": (
            "Crizotinib was evaluated as an ALK inhibitor in NCI-H2228 cells harboring the EML4-ALK fusion. "
            "A 72-hour CellTiter-Glo assay demonstrated an IC50 of 20 nM. Phospho-flow cytometry showed "
            "dose-dependent inhibition of ALK Y1604 phosphorylation. Crizotinib also inhibited MET and ROS1 "
            "kinases with IC50 values of 8 nM and 15 nM respectively in biochemical assays."
        ),
    },
    {
        "id": 5,
        "text": (
            "Ibrutinib is an irreversible covalent inhibitor of Bruton's tyrosine kinase (BTK). "
            "Biochemical assays using purified BTK protein demonstrated apparent IC50 of 0.5 nM. "
            "In the Ramos Burkitt lymphoma cell line, a 48-hour CTG assay showed an IC50 of 12 nM. "
            "Occupancy studies confirmed >95% BTK covalent engagement at 1 μM."
        ),
    },
    {
        "id": 6,
        "text": (
            "Palbociclib is a selective CDK4/6 inhibitor approved for HR+/HER2- breast cancer. "
            "In MCF-7 breast cancer cells, palbociclib inhibited proliferation with an IC50 of 66 nM "
            "in a 5-day BrdU incorporation assay. Biochemical IC50 values were 11 nM for CDK4/cyclin D1 "
            "and 16 nM for CDK6/cyclin D3. Rb phosphorylation at S780 was abolished above 100 nM."
        ),
    },
    {
        "id": 7,
        "text": (
            "Olaparib, a PARP1/2 inhibitor, was characterized in BRCA1-mutant MDA-MB-436 triple-negative breast "
            "cancer cells. A 6-day CellTiter-Glo assay revealed an IC50 of 9 nM, consistent with synthetic lethality "
            "in HRD-deficient cells. Biochemical PARP1 trapping assay showed an IC50 of 5 nM. In isogenic "
            "BRCA1-wild-type cells, the IC50 shifted to 4.2 μM (>400-fold), confirming BRCA1-dependent sensitivity."
        ),
    },
    {
        "id": 8,
        "text": (
            "Osimertinib (AZD9291) is a third-generation EGFR inhibitor designed to overcome T790M resistance. "
            "In PC-9 VanR cells (EGFR exon 19 del / T790M), a 72-hour CellTiter-Glo assay demonstrated an "
            "IC50 of 1.0 nM. Biochemical EGFR T790M kinase assay showed Ki = 0.4 nM. Against wild-type EGFR, "
            "the IC50 was 184 nM, giving >180-fold selectivity for the mutant form."
        ),
    },
]

print(f"✅ Loaded {len(abstracts)} abstracts for batch extraction.")


In [ ]:
# ============================================================
# STEP 9: Process all abstracts in batch
# ============================================================
# Each abstract is extracted independently.
# We store each record as a dict for easy DataFrame construction.

records = []

for item in abstracts:
    print(f"Extracting abstract {item['id']}...", end=" ", flush=True)
    try:
        record = extraction_chain.invoke({"abstract": item["text"]})
        records.append({
            "abstract_id": item["id"],
            "drug_name": record.drug_name,
            "target": record.target,
            "ic50_value": record.ic50_value,
            "ic50_unit": record.ic50_unit,
            "cell_line": record.cell_line,
            "assay_type": record.assay_type,
            "key_finding": record.key_finding,
        })
        print("done")
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\n✅ Extracted {len(records)} records successfully.")


In [ ]:
# ============================================================
# STEP 10: Display results as a pandas DataFrame
# ============================================================

import pandas as pd

df = pd.DataFrame(records)

# Display settings for readability
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 20)

print("Extracted IC50 Table:")
print(df[["drug_name", "target", "ic50_value", "ic50_unit", "cell_line", "assay_type"]].to_string(index=False))


In [ ]:
# ============================================================
# STEP 11: Query and filter the DataFrame
# ============================================================
# Now that data is structured, you can filter it like any table.

print("=== Compounds with IC50 < 5 nM (cell-based) ===")
potent = df[
    (df["ic50_unit"] == "nM") &
    (df["ic50_value"] < 5) &
    (df["cell_line"].notna())
][["drug_name", "target", "ic50_value", "ic50_unit", "cell_line"]]
print(potent.to_string(index=False))

print("\n=== CellTiter-Glo assays only ===")
ctg = df[df["assay_type"].str.contains("CellTiter|CTG", case=False, na=False)]
print(ctg[["drug_name", "target", "ic50_value", "ic50_unit"]].to_string(index=False))

print("\n=== Sort by IC50 (nM, ascending) ===")
nM_df = df[df["ic50_unit"] == "nM"].sort_values("ic50_value")
print(nM_df[["drug_name", "target", "ic50_value", "cell_line"]].to_string(index=False))


In [ ]:
# ============================================================
# STEP 12 (Optional): Export to CSV
# ============================================================

df.to_csv("extracted_ic50_table.csv", index=False)
print("✅ Saved to extracted_ic50_table.csv")
print("   You can download this from Colab: Files → extracted_ic50_table.csv")


---
## What You Built

You created an LLM-powered data extraction pipeline that:

1. **Defines a schema** with Pydantic — one class describes exactly what fields to extract
2. **Forces structured output** — the LLM fills in the fields, not a prose paragraph
3. **Processes batches** — runs over any number of abstracts programmatically
4. **Outputs a DataFrame** — immediately queryable, filterable, and exportable

This pattern works for any extraction task: clinical endpoints, patient demographics,
experimental conditions, gene lists — define a schema, run the chain.

---

## Scaling Up

To run this on your own literature:
- Replace the `abstracts` list with PubMed abstracts (use the `biopython` Entrez API to fetch them)
- Adjust the `DrugActivityRecord` schema fields to match what you need
- For unpublished internal data, swap `ChatGoogleGenerativeAI` for `ChatOllama` (Notebook 03 pattern)

---

## Next Step

**Notebook 06** — Combine everything into a multi-agent pipeline:
a literature agent + a database agent + a coordinator that writes a full target assessment report.
